In [40]:
# This script installs the necessary Python packages for data analysis and visualization.
%pip install pandas numpy matplotlib seaborn missingno scipy
%pip install openpyxl
%pip install pyblp


import pandas as pd 
import glob 
import re 
import numpy as np 
import re


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [41]:
################ Import all necessary data

# Initialize storage
combined_data = []

# Import Sales Data
file_paths = glob.glob("C:/Users/Lenovo/Desktop/Dissertaion/China Data/Demand/Sales/*.xlsx")  # Update path

for file in file_paths:
    # Extract year from filename (assuming format like "...2023.xlsx")
    year = re.search(r'\d{4}', file).group()  # Finds first 4-digit number
    
    # Read all sheets from current file
    sheets_dict = pd.read_excel(file, sheet_name=None)
    
    for sheet_name, df in sheets_dict.items():
        # Add identifier columns
        df['year'] = int(year)          # From filename
        df['type'] = sheet_name        # From sheet name
        
        combined_data.append(df)

# Combine all sales data
final_df = pd.concat(combined_data, ignore_index=True)

# Import Price Data
price_file = pd.read_excel(
	"C:/Users/Lenovo/Desktop/Dissertaion/China Data/Demand/Prices/Price 2015-202309.xlsx",
	sheet_name="中国汽车分车型每月销售量"
)

# Import Population Data
population_file = pd.read_excel(
    "C:/Users/Lenovo/Desktop/Dissertaion/China Data/Geographical Controls/(10)2000-2023年人口密度.xls",
    sheet_name=None
)

# Import charging station data
charging_station_paths = glob.glob("C:/Users/Lenovo/Desktop/Dissertaion/China Data/Charging/*.xlsx")

charging_station_data = []

for file in charging_station_paths:
    # Read all sheets from current file
    sheets_dict = pd.read_excel(file, sheet_name=None)
    for sheet_name, df in sheets_dict.items():
        # Optionally add file/sheet info
        df['source_file'] = file
        df['sheet_name'] = sheet_name
        charging_station_data.append(df)

# Combine all charging station data into one DataFrame
charging_station_df = pd.concat(charging_station_data, ignore_index=True)

# Extract 'year' from source_file and add as a column
charging_station_df['year'] = charging_station_df['source_file'].apply(
    lambda x: int(re.search(r'\d{4}', x).group()) if re.search(r'\d{4}', x) else np.nan
)

# Export charging station data to CSV
charging_station_df.to_csv("charging_station_data.csv", index=False)

# Import product characteristics data
characteristics = glob.glob("C:/Users/Lenovo/Desktop/Dissertaion/China Data/Demand/Characteristics/*.xlsx")

           

c:\Users\Lenovo\Desktop\Dissertaion\Code\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Lenovo\Desktop\Dissertaion\Code\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Lenovo\Desktop\Dissertaion\Code\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Lenovo\Desktop\Dissertaion\Code\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Lenovo\Desk

In [42]:
################ Process Price Data

# Define column mapping from Chinese to English
column_mapping = {
    '年份': 'year',
    '月份': 'month',
    '排名': 'rank',
    '车型': 'model',
    '厂商': 'manufacturer',
    '销量': 'sales',
    '售价（万元）': 'price',
    '燃油类型' : 'fuel_type',
}

# Rename columns to English first
price_file = price_file.rename(columns=column_mapping)

# Standardize column names to lowercase with underscores
def standardize_columns(df):
    df.columns = (df.columns
                 .str.lower()  # Convert to lowercase
                 .str.replace(' ', '_')  # Replace spaces with underscores
                 .str.replace('-', '_'))  # Replace hyphens with underscores
    return df

# Apply standardization to both DataFrames
price_file = standardize_columns(price_file)

# Drop irrelevant columns (keep only columns we need)
columns_to_keep_en = ['year', 'month', 'model', 'sales', 'price']
price_file = price_file[columns_to_keep_en].copy()

# Define function to convert price ranges to midpoints
def price_range_to_midpoint(price_str):
    if isinstance(price_str, str):
        # Remove any whitespace
        price_str = price_str.replace(' ', '')
        # Handle range like "9.98-17.98"
        if '-' in price_str:
            parts = price_str.split('-')
            try:
                low = float(parts[0])
                high = float(parts[1])
                return (low + high) / 2
            except ValueError:
                return np.nan
        try:
            return float(price_str)
        except ValueError:
            return np.nan
    return np.nan

# Convert price ranges to midpoints
price_file['price'] = price_file['price'].apply(price_range_to_midpoint)

# Calculate yearly weighted average price
def calculate_weighted_avg(group):
    total_sales = group['sales'].sum()
    weighted_sum = (group['sales'] * group['price']).sum()
    return weighted_sum / total_sales if total_sales != 0 else 0

yearly_weighted_prices = price_file.groupby(['model', 'year']).apply(calculate_weighted_avg).reset_index()
yearly_weighted_prices.columns = ['model', 'year', 'weighted_Avg_Price']

# Add total yearly sales for context
yearly_sales = price_file.groupby(['model', 'year'])['sales'].sum().reset_index()
yearly_weighted_prices = yearly_weighted_prices.merge(yearly_sales, on=['model', 'year'])

# Sort the results
yearly_weighted_prices = yearly_weighted_prices.sort_values(['model', 'year'])

# Display results
print(yearly_weighted_prices)

       model  year  weighted_Avg_Price  sales
0        212  2021               10.29   1433
1        212  2023               10.29   1037
2     AION S  2019               15.98  31929
3     AION S  2020               15.98  46091
4     AION S  2021               15.98  69220
...      ...   ...                 ...    ...
5348      魔方  2023               12.69   5871
5349       鲸  2022                0.00    463
5350       鲸  2023                0.00    170
5351     黑金刚  2015                0.00   1826
5352     黑金刚  2016                0.00    462

[5353 rows x 4 columns]


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_23772\2752139757.py:62: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  yearly_weighted_prices = price_file.groupby(['model', 'year']).apply(calculate_weighted_avg).reset_index()


In [43]:
################ Process Sales Data

# Translate variable names
# Manual translation
translation_map = {
    '省份': 'province',
    '品牌': 'brand',
    '车型': 'model',
    '燃料类型': 'fuel_type',
    '功率': 'power',
    '销量'  : 'sales',
    '总质量': 'mass',
}

# Apply translations
combined_df = final_df.rename(columns=translation_map)

# Standardize column names to lowercase with underscores
combined_df = standardize_columns(combined_df)

# Move 'year' and 'type' to the front
cols = ['year', 'type'] + [col for col in combined_df.columns if col not in ['year', 'type']]
combined_df = combined_df[cols]

# Converts data types
combined_df = combined_df.convert_dtypes()

# View variable names
print(combined_df.columns)


Index(['year', 'type', 'province', 'brand', 'model', 'fuel_type', 'mass',
       'power', 'sales'],
      dtype='object')


In [44]:
################ Combine Sales and Price Data

# Drop sales and month from price_df 
yearly_weighted_prices = yearly_weighted_prices.drop(columns=['sales'], errors='ignore')  # Ignore if 'sales' column doesn't exist

# Merge the datasets on model and year
merged_df = pd.merge(combined_df, yearly_weighted_prices, 
                    on=['model', 'year'], 
                    how='inner')  # Inner join to keep only matching models

# Verify if any models were dropped
original_models = set(combined_df['model'].unique())
merged_models = set(merged_df['model'].unique())
dropped_models = original_models - merged_models

if len(dropped_models) > 0:
    print(f"The following models were dropped due to missing price data: {dropped_models}")
else:
    print("All models had matching price data and were kept.")

# Display the first few rows of the combined data
print("\nCombined data preview:")
print(merged_df.head())

The following models were dropped due to missing price data: {'EQA', 'MG6', '力帆320', '君马MEET 3', '花冠', '昌河Q25', 'QX50', '沐飒', '东风小康K07S', 'A06', '宏瑞小虎 BEV', '艾瑞泽M7', 'C3-XR', '大7 MPV', '名爵Cyberster', '红旗H5 FCEV', '思铭X-NV', '北汽EU5', '凯迪拉克IQ傲歌', '大通MIFA 7', 'C11', '瑞虎7 PLUS', '星途VX', '大乘G70', 'DS9 PHEV', '蓝电E3 BEV', 'F3', '长安E-Pro', '跨越星V5', '微蓝 7', '传祺GA3S', '江淮IEV7S', 'EV系列', '海马S3', '幻速S6', '大众Polo', '睿行M60', '极氪007', 'ix35', '秦EV', 'DS 5LS', '绅宝D60', '比亚迪秦Pro', '上汽大通G20', '上汽大通G10', '宋Pro', '艾瑞泽7', '风光580 PLUS', '广汽Aion S', '江淮iEV7S', 'ES6', '思皓爱跑S', '艾瑞泽EX', '讴歌TLX-L', '传祺影豹', '大众ID.6 X', '斯威大虎', '云兔', '荣威ei6 MAX', '比亚迪G5', '幸福e+', 'MG MULAN', 'E6', 'MG5', '景逸X3', '比亚迪S7', '宝骏kiwi', '江淮iEVS4', '别克Electra E4', 'EQE SUV', '秦', '北汽E系列', 'iEV7L', '森雅R9', '问界M5 EV', '长安启源Q05', '众泰T600 Coupe', '风光E380', '几何C', '合创V09', '风行Friday', 'CS95', '江南T11', '奔奔mini', '哈弗H7L', '凯翼C3', 'MG EZS', 'T60', '北汽ET3', '极星4', '幻速S3', 'ARCFOX αS', '五菱宏光MINI EV 敞篷版', '现代ix25', '风神皓极', 'HYCAN 007', '君阁', '昌河Q7'

In [45]:
################ Process Population Data
# Select the correct sheet from the dictionary (replace 'Sheet1' with the actual sheet name if needed)
population_df = population_file['Sheet1'].copy()

# Convert poplation_df to dataframe
population_df = pd.DataFrame(population_df)

In [46]:
################ Cleaning Merged Data

# Converts mass to numeric, handling potential string values or missing data
merged_df['mass'] = pd.to_numeric(merged_df['mass'], errors='coerce').astype(float)

# Function to clean power values
def clean_power(power):
    if isinstance(power, str):
        # Extract all numbers from the string
        numbers = [int(s) for s in power.replace(',', ' ').split() if s.isdigit()]
        if numbers:
            return np.mean(numbers)  # Return average if multiple numbers
        else:
            return np.nan
    return power

# Clean the power column
merged_df['power'] = merged_df['power'].apply(clean_power)

# Merge models, compute weighted average power and mass
# Group by all columns except mass, power, sales, and weighted_Avg_Price
group_cols = ['year', 'type', 'province', 'brand', 'model', 'fuel_type']
other_cols = ['mass', 'power', 'sales', 'weighted_Avg_Price']

# Calculate sales-weighted averages for mass and power
def weighted_average(group):
    sales = group['sales']
    total_sales = sales.sum()
    
    # Calculate weighted averages
    weighted_mass = (group['mass'] * sales).sum() / total_sales
    weighted_power = (group['power'] * sales).sum() / total_sales
    
    # Sum sales and take first weighted_Avg_Price (should be same within group)
    return pd.Series({
        'mass': weighted_mass,
        'power': weighted_power,
        'sales': total_sales,
        'weighted_Avg_Price': group['weighted_Avg_Price'].iloc[0] 
    })

# Apply the grouping and weighted average calculation
merged_df = merged_df.groupby(group_cols).apply(weighted_average).reset_index()

# Classify fuel types
def classify_fuel_type(name):
    name = str(name) 
    if '混合动力' in name:
        return 'PHEV'
    elif '纯电动' in name:
        return 'BEV'
    else:
        return 'Gas'
    
# Apply classification 
merged_df['fuel_type'] = merged_df['fuel_type'].apply(classify_fuel_type)

# Display results
print(f"\nMerged shape: {merged_df.shape}")
print("\nFirst few merged rows:")
display(merged_df.head())




Merged shape: (79290, 10)

First few merged rows:


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_23772\3358788437.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  merged_df = merged_df.groupby(group_cols).apply(weighted_average).reset_index()


,year,type,province,brand,model,fuel_type,mass,power,sales,weighted_Avg_Price
0,2019,国产新能源乘用车,上海市,东风风行,景逸S50,BEV,2036.0,90.0,1.0,7.49
1,2019,国产新能源乘用车,上海市,丰田,卡罗拉,PHEV,1975.0,53.0,665.0,13.48
2,2019,国产新能源乘用车,上海市,丰田,雷凌,PHEV,1975.0,53.0,1681.0,13.23
3,2019,国产新能源乘用车,上海市,云度,云度π1,BEV,1785.0,90.0,1.0,7.73
4,2019,国产新能源乘用车,上海市,云度,云度π3,BEV,1845.0,90.0,75.0,13.23


In [47]:
################ Handle missing values and zeros

# Check the initial shape of the merged dataframe
initial_rows = merged_df.shape[0]
print(f"Initial number of rows in merged_df: {initial_rows}")

# Identify missing values in merged_df
missing_values = merged_df.isnull().sum()
print("\nMissing values per column in merged_df:")
print(missing_values)

# Identify zeros in numeric columns of merged_df
# Select numeric columns (excluding year which shouldn't have zeros)
numeric_cols = merged_df.select_dtypes(include=['int64', 'float64']).columns
numeric_cols = [col for col in numeric_cols if col != 'year']

zeros_count = {}
for col in numeric_cols:
    zeros_count[col] = (merged_df[col] == 0).sum()

print("\nZero values in numeric columns of merged_df:")
print(pd.Series(zeros_count))

# Remove rows with missing values or zeros in key columns
# Define which columns to check for zeros (mass, power, sales, weighted_Avg_Price)
cols_to_check = ['mass', 'power', 'sales', 'weighted_Avg_Price']

# Create a mask for rows to keep (no NA and no zeros in specified columns)
mask = (~merged_df[cols_to_check].isnull()).all(axis=1)
for col in cols_to_check:
    mask &= (merged_df[col] != 0)

# Apply the filter to create cleaned_merged_df
cleaned_merged_df = merged_df[mask].copy()

# Report results
removed_rows = initial_rows - cleaned_merged_df.shape[0]
print(f"\nTotal rows removed from merged_df: {removed_rows}")
print(f"New cleaned_merged_df shape: {cleaned_merged_df.shape}")

# Breakdown of why rows were removed
print("\nBreakdown of removed rows:")
print(f"- Rows with missing values: {initial_rows - (~merged_df.isnull()).all(axis=1).sum()}")
for col in cols_to_check:
    print(f"- Rows with zero in {col}: {(merged_df[col] == 0).sum()}")

Initial number of rows in merged_df: 79290

Missing values per column in merged_df:
year                  0
type                  0
province              0
brand                 0
model                 0
fuel_type             0
mass                  0
power                 0
sales                 0
weighted_Avg_Price    0
dtype: int64

Zero values in numeric columns of merged_df:
mass                     2
power                  565
sales                    0
weighted_Avg_Price    7915
dtype: int64

Total rows removed from merged_df: 8399
New cleaned_merged_df shape: (70891, 10)

Breakdown of removed rows:
- Rows with missing values: 0
- Rows with zero in mass: 2
- Rows with zero in power: 565
- Rows with zero in sales: 0
- Rows with zero in weighted_Avg_Price: 7915


In [48]:
merged_df = cleaned_merged_df

In [49]:
################ Calculating market share using population as total market size
AVERAGE_HOUSEHOLD_SIZE = 2.6

# Calculate market size (in 10,000 households)
population_df['market_size'] = population_df['年末常住人口(万人)'] / AVERAGE_HOUSEHOLD_SIZE

# Merge market size data
merged_df = merged_df.merge(
    population_df[['省份名称', 'year', 'market_size']],
    how='left',
    left_on=['province', 'year'],
    right_on=['省份名称', 'year']
)

# Clean up after merge - drop the redundant province name column
merged_df = merged_df.drop(columns=['省份名称'])

# Convert to actual household count
merged_df['market_size'] = merged_df['market_size'] * 10000 

# Calculate market share with error handling
merged_df['market_share'] = merged_df['sales'] / merged_df['market_size']



In [50]:
# Calculate within-group shares (before any data loss in subsequent steps)
print("="*60)
print("CALCULATING WITHIN-GROUP SHARES")
print("="*60)

# Define nesting structure for vehicles
def assign_nesting_group(row):
    """
    Assign vehicles to nesting groups based on fuel type
    BEV (Battery Electric Vehicle): Pure electric
    PHEV (Plug-in Hybrid Electric Vehicle): Hybrid electric 
    Gas: Traditional gasoline vehicles
    """
    fuel_type = row['fuel_type']
    if fuel_type == 'BEV':
        return 1  # Electric nest
    elif fuel_type == 'PHEV': 
        return 1  # Electric nest (can also be separate nest if needed)
    else:  # Gas
        return 0  # Non-electric nest
        
# Apply nesting assignment
merged_df['nesting_ids'] = merged_df.apply(assign_nesting_group, axis=1)

# Create is_electric indicator
merged_df['is_electric'] = (merged_df['nesting_ids'] == 1).astype(int)

# Calculate market shares using market_size instead of total sales
merged_df['shares'] = merged_df['sales'] / merged_df['market_size']

# Calculate nest-level shares (total share of each nest in each market)
merged_df['nest_total_sales'] = merged_df.groupby(['province', 'year', 'nesting_ids'])['sales'].transform('sum')
merged_df['nest_shares'] = merged_df['nest_total_sales'] / merged_df['market_size']

# Calculate within-nest shares (share of each product within its nest)
merged_df['within_nest_shares'] = merged_df['sales'] / merged_df['nest_total_sales']

# Calculate EV_share as sum of all EV model shares in each market
merged_df['EV_total_sales'] = merged_df.groupby(['province', 'year'])['sales'].transform(
    lambda x: x[merged_df.loc[x.index, 'is_electric'] == 1].sum()
)
merged_df['EV_share'] = merged_df['EV_total_sales'] / merged_df['market_size']

# Display summary statistics
print("\nNesting Group Summary:")
print(merged_df.groupby(['nesting_ids', 'fuel_type']).size().reset_index(name='count'))

print(f"\nTotal observations: {len(merged_df):,}")
print(f"Electric vehicles (nesting_ids=1): {(merged_df['nesting_ids']==1).sum():,}")
print(f"Non-electric vehicles (nesting_ids=0): {(merged_df['nesting_ids']==0).sum():,}")

print("\nMarket Share Statistics:")
print(f"Market shares - Range: [{merged_df['shares'].min():.8f}, {merged_df['shares'].max():.8f}]")
print(f"Market shares - Mean: {merged_df['shares'].mean():.6f}")

print("\nWithin-Nest Share Statistics:")
print(f"Within-nest shares - Range: [{merged_df['within_nest_shares'].min():.8f}, {merged_df['within_nest_shares'].max():.8f}]")
print(f"Within-nest shares - Mean: {merged_df['within_nest_shares'].mean():.6f}")

# Check for potential issues
very_small_within_nest = (merged_df['within_nest_shares'] < 0.001).sum()
very_large_within_nest = (merged_df['within_nest_shares'] > 0.9).sum()

print(f"\nPotential Issues:")
print(f"Within-nest shares < 0.001: {very_small_within_nest:,} ({100*very_small_within_nest/len(merged_df):.2f}%)")
print(f"Within-nest shares > 0.9: {very_large_within_nest:,} ({100*very_large_within_nest/len(merged_df):.2f}%)")

# Verify shares sum correctly
print(f"\nVerification:")
market_share_sums = merged_df.groupby(['province', 'year'])['shares'].sum()
print(f"Market shares using market_size - Range: [{market_share_sums.min():.6f}, {market_share_sums.max():.6f}]")
print(f"Market shares using market_size - Mean: {market_share_sums.mean():.6f}")

within_nest_sums = merged_df.groupby(['province', 'year', 'nesting_ids'])['within_nest_shares'].sum()
print(f"Within-nest shares sum to 1.0 within each nest: {np.allclose(within_nest_sums, 1.0)}")

# Verify EV_share calculation
print(f"\nEV Share Statistics:")
print(f"EV_share - Range: [{merged_df['EV_share'].min():.6f}, {merged_df['EV_share'].max():.6f}]")
print(f"EV_share - Mean: {merged_df['EV_share'].mean():.6f}")
print(f"EV_share - Median: {merged_df['EV_share'].median():.6f}")

# Check markets with zero EV share
zero_ev_markets = (merged_df['EV_share'] == 0).sum()
print(f"Markets with zero EV share: {zero_ev_markets:,} observations")

# Show sample of the new columns
print(f"\nSample of new columns:")
sample_cols = ['province', 'year', 'model', 'fuel_type', 'nesting_ids', 'is_electric', 
               'sales', 'market_size', 'shares', 'within_nest_shares', 'EV_share']
print(merged_df[sample_cols].head(10))

print("="*60)

CALCULATING WITHIN-GROUP SHARES

Nesting Group Summary:
   nesting_ids fuel_type  count
0            0       Gas  46759
1            1       BEV  10357
2            1      PHEV  13775

Total observations: 70,891
Electric vehicles (nesting_ids=1): 24,132
Non-electric vehicles (nesting_ids=0): 46,759

Market Share Statistics:
Market shares - Range: [0.00000002, 0.00446157]
Market shares - Mean: 0.000055

Within-Nest Share Statistics:
Within-nest shares - Range: [0.00000062, 0.56130775]
Within-nest shares - Mean: 0.004373

Potential Issues:
Within-nest shares < 0.001: 34,782 (49.06%)
Within-nest shares > 0.9: 0 (0.00%)

Verification:
Market shares using market_size - Range: [0.014213, 0.047667]
Market shares using market_size - Mean: 0.025259
Within-nest shares sum to 1.0 within each nest: True

EV Share Statistics:
EV_share - Range: [0.000081, 0.019526]
EV_share - Mean: 0.003669
EV_share - Median: 0.002397
Markets with zero EV share: 0 observations

Sample of new columns:
  province  yea

In [51]:
################ Generate IDs for provinces, brands, models, and products

# Create province_id (leading zeros for 2 digits)
merged_df['province_id'] = pd.factorize(merged_df['province'])[0] + 1
merged_df['province_id'] = merged_df['province_id'].apply(lambda x: f"P{x:02d}")

# Create brand_id (leading zeros for 2 digits)
merged_df['brand_id'] = pd.factorize(merged_df['brand'])[0] + 1
merged_df['brand_id'] = merged_df['brand_id'].apply(lambda x: f"B{x:02d}")

# Create model_id within each brand (leading zeros for 2 digits)
merged_df['model_id'] = merged_df.groupby('brand')['model'].transform(lambda x: pd.factorize(x)[0] + 1)
merged_df['model_id'] = merged_df['model_id'].apply(lambda x: f"M{x:02d}")

# Create product_id by combining brand_id and model_id
merged_df['product_id'] = merged_df['brand_id'] + merged_df['model_id']

# Create market_id in format P##Y####
merged_df['market_id'] = merged_df['province_id'] + 'Y' + merged_df['year'].astype(str)

# Create mapping tables for reference
province_mapping = merged_df[['province', 'province_id']].drop_duplicates().sort_values('province_id')
brand_mapping = merged_df[['brand', 'brand_id']].drop_duplicates().sort_values('brand_id')

# Add brand_id to model_mapping for sorting
model_mapping = merged_df[['brand', 'model', 'model_id', 'product_id']].drop_duplicates()
model_mapping = model_mapping.merge(brand_mapping, on='brand', how='left')
model_mapping = model_mapping.sort_values(['brand_id', 'model_id'])
market_mapping = merged_df[['province', 'year', 'market_id']].drop_duplicates().sort_values('market_id')

# Display the mappings
print("Province Mapping:")
display(province_mapping.head())

print("\nBrand Mapping:")
display(brand_mapping.head())

print("\nModel/Product Mapping:")
display(model_mapping.head())

print("\nMarket Mapping:")
display(market_mapping.head())

# Show the updated dataframe with new ID columns
print("\nDataframe with new ID columns:")
display(merged_df.head())

Province Mapping:


,province,province_id
0,上海市,P01
46,云南省,P02
85,内蒙古自治区,P03
119,北京市,P04
164,吉林省,P05



Brand Mapping:


,brand,brand_id
0,东风风行,B01
1,丰田,B02
3,云度,B03
5,传祺,B04
6,凯迪拉克,B05



Model/Product Mapping:


,brand,model,model_id,product_id,brand_id
0,东风风行,景逸S50,M01,B01M01,B01
46,东风风行,菱智,M02,B01M02,B01
75,东风风行,风行T5,M03,B01M03,B01
397,东风风行,风光E1,M04,B01M04,B01
408,东风风行,风光E3,M05,B01M05,B01



Market Mapping:


,province,year,market_id
0,上海市,2019,P01Y2019
9767,上海市,2020,P01Y2020
21768,上海市,2021,P01Y2021
36236,上海市,2022,P01Y2022
50166,上海市,2023,P01Y2023



Dataframe with new ID columns:


,year,type,province,brand,model,fuel_type,mass,power,sales,weighted_Avg_Price,...,nest_total_sales,nest_shares,within_nest_shares,EV_total_sales,EV_share,province_id,brand_id,model_id,product_id,market_id
0,2019,国产新能源乘用车,上海市,东风风行,景逸S50,BEV,2036.0,90.0,1.0,7.49,...,29894.0,0.003133,0.000033,29894.0,0.003133,P01,B01,M01,B01M01,P01Y2019
1,2019,国产新能源乘用车,上海市,丰田,卡罗拉,PHEV,1975.0,53.0,665.0,13.48,...,29894.0,0.003133,0.022245,29894.0,0.003133,P01,B02,M01,B02M01,P01Y2019
2,2019,国产新能源乘用车,上海市,丰田,雷凌,PHEV,1975.0,53.0,1681.0,13.23,...,29894.0,0.003133,0.056232,29894.0,0.003133,P01,B02,M02,B02M02,P01Y2019
3,2019,国产新能源乘用车,上海市,云度,云度π1,BEV,1785.0,90.0,1.0,7.73,...,29894.0,0.003133,0.000033,29894.0,0.003133,P01,B03,M01,B03M01,P01Y2019
4,2019,国产新能源乘用车,上海市,云度,云度π3,BEV,1845.0,90.0,75.0,13.23,...,29894.0,0.003133,0.002509,29894.0,0.003133,P01,B03,M02,B03M02,P01Y2019


In [54]:
################ Process charging station and car stock data
# Remove duplicate 'province' columns if present
if charging_station_df.columns.tolist().count('province') > 1:
    charging_station_df = charging_station_df.loc[:, ~charging_station_df.columns.duplicated()]

# Rename charging station data to merged_df
charging_station_df = charging_station_df.rename(columns={
    '省份': 'province',
    '公共充电桩保有量（台）': 'charging_stations_stock'
})

# Keep only province, year, and charging station stock columns
charging_station_df = charging_station_df[['province', 'year', 'charging_stations_stock']]

# Ensure correct sorting before shifting
charging_station_df = charging_station_df.sort_values(['province', 'year'])

# Add lagged charging station stock (previous year for each province)
charging_station_df['charging_stations_stock_lag'] = charging_station_df.groupby('province')['charging_stations_stock'].shift(1)

# Filter for years 2018-2023 if needed
charging_station_df = charging_station_df[charging_station_df['year'].between(2018, 2023)]

# 移除 province 列为 NaN 的行，然后显示所有 charging_station_df 条目
charging_station_df = charging_station_df[charging_station_df['province'].notna()]


In [55]:
################ Merging charging station data with sales data

# Create a mapping dictionary between the two naming conventions
province_mapping = {
    '北京': '北京市',
    '天津': '天津市',
    '河北': '河北省',
    '山西': '山西省',
    '内蒙古': '内蒙古自治区',
    '辽宁': '辽宁省',
    '吉林': '吉林省',
    '黑龙江': '黑龙江省',
    '上海': '上海市',
    '江苏': '江苏省',
    '浙江': '浙江省',
    '安徽': '安徽省',
    '福建': '福建省',
    '江西': '江西省',
    '山东': '山东省',
    '河南': '河南省',
    '湖北': '湖北省',
    '湖南': '湖南省',
    '广东': '广东省',
    '广西': '广西壮族自治区',
    '海南': '海南省',
    '重庆': '重庆市',
    '四川': '四川省',
    '贵州': '贵州省',
    '云南': '云南省',
    '西藏': '西藏自治区',
    '陕西': '陕西省',
    '甘肃': '甘肃省',
    '青海': '青海省',
    '宁夏': '宁夏回族自治区',
    '新疆': '新疆维吾尔自治区',
    '香港': '香港特别行政区' 
}

# Apply the mapping to create a new column
charging_station_df['province'] = charging_station_df['province'].map(province_mapping)

# Drop existing charging station columns to avoid duplicate columns after merge
cols_to_drop = [col for col in merged_df.columns if col.startswith('charging_stations_stock')]
merged_df = merged_df.drop(columns=cols_to_drop, errors='ignore')

# Merge again
merged_df = pd.merge(merged_df, charging_station_df, left_on=['province', 'year'], right_on=['province', 'year'], how='left')

# Display the transformed data
merged_df.head()

,year,type,province,brand,model,fuel_type,mass,power,sales,weighted_Avg_Price,...,within_nest_shares,EV_total_sales,EV_share,province_id,brand_id,model_id,product_id,market_id,charging_stations_stock,charging_stations_stock_lag
0,2019,国产新能源乘用车,上海市,东风风行,景逸S50,BEV,2036.0,90.0,1.0,7.49,...,0.000033,29894.0,0.003133,P01,B01,M01,B01M01,P01Y2019,55113.0,39303.0
1,2019,国产新能源乘用车,上海市,丰田,卡罗拉,PHEV,1975.0,53.0,665.0,13.48,...,0.022245,29894.0,0.003133,P01,B02,M01,B02M01,P01Y2019,55113.0,39303.0
2,2019,国产新能源乘用车,上海市,丰田,雷凌,PHEV,1975.0,53.0,1681.0,13.23,...,0.056232,29894.0,0.003133,P01,B02,M02,B02M02,P01Y2019,55113.0,39303.0
3,2019,国产新能源乘用车,上海市,云度,云度π1,BEV,1785.0,90.0,1.0,7.73,...,0.000033,29894.0,0.003133,P01,B03,M01,B03M01,P01Y2019,55113.0,39303.0
4,2019,国产新能源乘用车,上海市,云度,云度π3,BEV,1845.0,90.0,75.0,13.23,...,0.002509,29894.0,0.003133,P01,B03,M02,B03M02,P01Y2019,55113.0,39303.0


In [56]:
################ Export the final cleaned dataframe to Excel
output_excel_path = 'final_cleaned_data.xlsx'

merged_df.to_excel('final_cleaned_data.xlsx', index=False)